# Auditoría Automatizada de Calidad de Datos y Arquitectura Medallion – PERÚ COMPRAS (2026)
**Autor:** Erick Otoya | **Stack:** Databricks, PySpark, Delta Lake, Unity Catalog  
**Fuente de datos:** Plataforma Nacional de Datos Abiertos del Perú – Central de Compras Públicas (PERÚ COMPRAS, periodos 2026-08 a 2026-10).

## 1. Capa Bronze: Ingesta de Datos Crudos y Trazabilidad
En esta etapa consolidé los archivos mensuales de órdenes de compra desde Unity Catalog Volumes sin inferir esquemas (**inferSchema=False**), para que los registros corruptos no detengan la carga. Agregué metadatos técnicos de auditoría (**fecha_auditoria y archivo_origen**) para poder rastrear el origen de cada registro.


In [0]:
from pyspark.sql import functions as F
import unicodedata

RUTA_ORIGEN = "/Volumes/workspace/default/peru_compras/"

df_raw = spark.read.csv(RUTA_ORIGEN, header=True, sep=";", inferSchema=False, encoding="ISO-8859-1")

# Buena práctica: Estandarizar nombres de columnas eliminando tildes (ej. ORDEN_ELECTRÓNICA -> ORDEN_ELECTRONICA)
def quitar_tildes(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

nuevas_columnas = [quitar_tildes(col) for col in df_raw.columns]
df_bronze = (
    df_raw.toDF(*nuevas_columnas)
    .withColumn("fecha_auditoria", F.current_timestamp())
    .withColumn("archivo_origen", F.col("_metadata.file_name"))
)

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_peru_compras")

print(f"Total de registros consolidados en Capa Bronze: {df_bronze.count()}")
print("Columnas estandarizadas:", df_bronze.columns)


Total de registros consolidados en Capa Bronze: 17623
Columnas estandarizadas: ['FECHA_PROCESO', 'RUC_PROVEEDOR', 'PROVEEDOR', 'RUC_ENTIDAD', 'ENTIDAD', 'TIPO_PROCEDIMIENTO', 'ORDEN_ELECTRONICA', 'ORDEN_ELECTRONICA_GENERADA', 'ESTADO_ORDEN_ELECTRONICA', 'DOCUMENTO_ESTADO_OCAM', 'FECHA_FORMALIZACION', 'FECHA_ULTIMO_ESTADO', 'SUB_TOTAL', 'IGV', 'TOTAL', 'ORDEN_DIGITALIZADA', 'DESCRIPCION_ESTADO', 'DESCRIPCION_CESION_DERECHOS', 'ACUERDO_MARCO', 'fecha_auditoria', 'archivo_origen']


## 2. Motor de Reglas de Calidad (Marco DAMA-DMBOK2)
Evalué la integridad, validez y consistencia de cada orden de compra antes de pasarla a Silver. En lugar de descartar registros en silencio, cada fila defectuosa guarda una lista (`ARRAY`) con todas las reglas que incumple:
1. **Unicidad (`ORDEN_DUPLICADA_O_NULA`):** detecta órdenes electrónicas repetidas o nulas en el consolidado.
2. **Validez de RUC (`RUC_PROVEEDOR_INVALIDO` / `RUC_ENTIDAD_INVALIDO`):** verifica el formato: 11 dígitos. El RUC del proveedor debe iniciar en 10 o 20, y el de la entidad debe iniciar en 20.
3. **Consistencia financiera (`INCONSISTENCIA_FINANCIERA`):** valida que `SUB_TOTAL + IGV` coincida con el `TOTAL` declarado (tolerancia de S/ 0.10 por redondeo) y que el monto sea mayor a cero.
4. **Integridad temporal (`FECHA_INCONSISTENTE`):** comprueba que `FECHA_ULTIMO_ESTADO` no sea anterior a `FECHA_FORMALIZACION`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.default.bronze_peru_compras")

def evaluar_calidad(df_entrada):
    ventana_orden = Window.partitionBy("ORDEN_ELECTRONICA")
    return (
        df_entrada
        .withColumn("conteo_orden", F.count("ORDEN_ELECTRONICA").over(ventana_orden))
        .withColumn("sub_total_num", F.col("SUB_TOTAL").cast("double"))
        .withColumn("igv_num", F.col("IGV").cast("double"))
        .withColumn("total_num", F.col("TOTAL").cast("double"))
        .withColumn("f_formalizacion", F.to_date(F.substring("FECHA_FORMALIZACION", 1, 10)))
        .withColumn("f_ultimo_estado", F.to_date(F.substring("FECHA_ULTIMO_ESTADO", 1, 10)))
        .withColumn(
            "motivos_rechazo",
            F.array_compact(F.array(
                F.when(F.col("ORDEN_ELECTRONICA").isNull() | (F.col("conteo_orden") > 1), F.lit("ORDEN_DUPLICADA_O_NULA")),
                F.when(F.col("RUC_PROVEEDOR").isNull() | (~F.col("RUC_PROVEEDOR").rlike("^(10|20)\\d{9}$")), F.lit("RUC_PROVEEDOR_INVALIDO")),
                F.when(F.col("RUC_ENTIDAD").isNull() | (~F.col("RUC_ENTIDAD").rlike("^20\\d{9}$")), F.lit("RUC_ENTIDAD_INVALIDO")),
                F.when(
                    F.col("total_num").isNull() | (F.col("total_num") <= 0) |
                    (F.abs((F.coalesce("sub_total_num", F.lit(0)) + F.coalesce("igv_num", F.lit(0))) - F.col("total_num")) > 0.10),
                    F.lit("INCONSISTENCIA_FINANCIERA")
                ),
                F.when(
                    F.col("f_formalizacion").isNull() | F.col("f_ultimo_estado").isNull() | (F.col("f_ultimo_estado") < F.col("f_formalizacion")),
                    F.lit("FECHA_INCONSISTENTE")
                )
            ))
        )
    )

# FASE A: Auditoría sobre datos 100% reales (Baseline limpio)
df_auditoria_real = evaluar_calidad(df)
total_df = df.count()
rechazos_reales = df_auditoria_real.filter(F.size("motivos_rechazo") > 0).count()
print(f"1. Auditoría Fuente Real (Sin inyección): {rechazos_reales} registros observados de {total_df} ({rechazos_reales / total_df * 100:.2f}% tasa de error en origen).")

# FASE B: Negative Testing determinístico (ordenado por ORDEN_ELECTRONICA) preservando TOTAL_REAL
ventana_secuencial = Window.orderBy("ORDEN_ELECTRONICA")

df_test_negativo = (
    df.withColumn("seq_id", F.row_number().over(ventana_secuencial))
    .withColumn("TOTAL_REAL", F.col("TOTAL").cast("double"))  # Guardamos el monto real sin distorsión
    .withColumn("RUC_PROVEEDOR", F.when(F.col("seq_id") % 50 == 0, F.lit("99900011122")).otherwise(F.col("RUC_PROVEEDOR")))
    .withColumn("TOTAL", F.when(F.col("seq_id") % 65 == 0, F.lit("-120.00")).otherwise(F.col("TOTAL")))
    .withColumn("FECHA_ULTIMO_ESTADO", F.when(F.col("seq_id") % 80 == 0, F.lit("2020-01-01")).otherwise(F.col("FECHA_ULTIMO_ESTADO")))
)

df_auditado = evaluar_calidad(df_test_negativo)
total_aislados = df_auditado.filter(F.size("motivos_rechazo") > 0).count()
print(f"2. Negative Testing (Inyección controlada): {total_aislados} registros aislados ({total_aislados / total_df * 100:.2f}% de los registros).")

1. Auditoría Fuente Real (Sin inyección): 0 registros observados de 17623 (0.00% tasa de error en origen).


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


2. Negative Testing (Inyección controlada): 759 registros aislados (4.31% de los registros).


## 3. Bifurcación Automatizada: Capa Silver vs. Capa Quarantine (Cuarentena)

Separé los registros según el resultado de las reglas de calidad, en dos corridas:

**Corrida real (producción)**
* **Capa Silver (`silver_peru_compras`):** registros reales que superaron el 100% de las validaciones, con tipos de datos estandarizados.
* **Capa Quarantine (`quarantine_peru_compras`):** registros reales con anomalías, junto con sus etiquetas de rechazo, para revisión y análisis de causa raíz (RCA).

**Prueba negativa**
* **`silver_peru_compras_test` y `quarantine_peru_compras_test`:** mismas capas, pero generadas con anomalías inyectadas a propósito. Las guardé aparte para no mezclarlas con los datos reales.

In [0]:
from pyspark.sql import functions as F

columnas_auxiliares = ["conteo_orden", "sub_total_num", "igv_num", "total_num", "f_formalizacion", "f_ultimo_estado"]

def construir_silver(df_aud, extra_drop=[]):
    return (
        df_aud
        .filter(F.size("motivos_rechazo") == 0)
        .withColumn("SUB_TOTAL", F.col("sub_total_num"))
        .withColumn("IGV", F.col("igv_num"))
        .withColumn("TOTAL", F.col("total_num"))
        .withColumn("FECHA_FORMALIZACION", F.col("f_formalizacion"))
        .withColumn("FECHA_ULTIMO_ESTADO", F.col("f_ultimo_estado"))
        .drop(*(columnas_auxiliares + ["motivos_rechazo"] + extra_drop))
    )

# ---------- PRODUCCIÓN: datos 100% reales ----------
df_silver = construir_silver(df_auditoria_real)
df_cuarentena = df_auditoria_real.filter(F.size("motivos_rechazo") > 0).drop(*columnas_auxiliares)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_peru_compras")
df_cuarentena.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.quarantine_peru_compras")

# ---------- PRUEBA NEGATIVA: tablas separadas con sufijo _test ----------
df_silver_test = construir_silver(df_auditado, extra_drop=["seq_id", "TOTAL_REAL"])
df_cuarentena_test = df_auditado.filter(F.size("motivos_rechazo") > 0).drop(*(columnas_auxiliares + ["seq_id"]))

df_silver_test.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_peru_compras_test")
df_cuarentena_test.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.quarantine_peru_compras_test")

total_real = df_auditoria_real.count()
print("=== CORRIDA REAL (producción) ===")
print(f"Auditados: {total_real} | Silver: {df_silver.count()} | Cuarentena: {df_cuarentena.count()}")
print("=== PRUEBA NEGATIVA (tablas _test) ===")
print(f"Silver_test: {df_silver_test.count()} | Cuarentena_test: {df_cuarentena_test.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


=== CORRIDA REAL (producción) ===
Auditados: 17623 | Silver: 17623 | Cuarentena: 0
=== PRUEBA NEGATIVA (tablas _test) ===
Silver_test: 16864 | Cuarentena_test: 759


## 4. Capa Gold: Resumen de Calidad y Prueba Negativa

Construí tres tablas Gold:

* **`gold_resumen_calidad_peru_compras`:** resumen de la corrida real, con los registros auditados, aprobados y observados.
* **`gold_scorecard_calidad_peru_compras`:** desglose (`explode`) de las listas de rechazo de los datos reales, para medir cuántas veces falla cada regla y cuánto dinero en soles está asociado. Si la fuente no incumple ninguna regla, queda vacía.
* **`gold_scorecard_prueba_negativa`:** mismo desglose, pero sobre las anomalías que inyecté a propósito. Uso `TOTAL_REAL` para que el monto no se distorsione con los valores falsos.

Al final comparo las anomalías esperadas con las detectadas para medir si el motor de reglas realmente funciona.

In [0]:
from pyspark.sql import functions as F

# 1) Resumen ejecutivo de la corrida real (1 fila)
resumen = spark.createDataFrame(
    [(df_auditoria_real.count(), df_silver.count(), df_cuarentena.count())],
    "registros_auditados long, aprobados long, observados long"
)
resumen.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_resumen_calidad_peru_compras")

# 2) Scorecard de anomalías REALES (queda vacío si la fuente no incumple ninguna regla)
gold_real = (
    spark.table("workspace.default.quarantine_peru_compras")
    .withColumn("tipo_anomalia", F.explode("motivos_rechazo"))
    .groupBy("tipo_anomalia")
    .agg(
        F.count("*").alias("registros_afectados"),
        F.round(F.sum(F.coalesce(F.col("TOTAL").cast("double"), F.lit(0.0))), 2).alias("monto_observado_soles")
    )
    .orderBy(F.desc("registros_afectados"))
)
gold_real.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_scorecard_calidad_peru_compras")

# 3) Scorecard de la PRUEBA NEGATIVA (anomalías inyectadas, usa TOTAL_REAL)
gold_test = (
    spark.table("workspace.default.quarantine_peru_compras_test")
    .withColumn("tipo_anomalia", F.explode("motivos_rechazo"))
    .groupBy("tipo_anomalia")
    .agg(
        F.count("*").alias("registros_detectados"),
        F.round(F.sum(F.coalesce(F.col("TOTAL_REAL"), F.lit(0.0))), 2).alias("monto_real_afectado_soles")
    )
    .orderBy(F.desc("registros_detectados"))
)
gold_test.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_scorecard_prueba_negativa")

# 4) Métrica de detección: esperado (aritmética) vs detectado
n = df_auditoria_real.count()
esperados = (n//50) + (n//65) + (n//80) - (n//650) - (n//400) - (n//1040) + (n//5200)
detectados = df_cuarentena_test.count()
print(f"Inyectados esperados: {esperados} | Detectados: {detectados} | Detección: {detectados/esperados*100:.1f}%")

display(resumen)
display(gold_test)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inyectados esperados: 759 | Detectados: 759 | Detección: 100.0%


registros_auditados,aprobados,observados
17623,17623,0


tipo_anomalia,registros_detectados,monto_real_afectado_soles
RUC_PROVEEDOR_INVALIDO,352,5344686.96
INCONSISTENCIA_FINANCIERA,271,7089340.64
FECHA_INCONSISTENTE,220,3979248.92


## 5. Gobierno de Datos y Metadatos en Unity Catalog
Documenté las tablas y los campos críticos directamente en el catálogo de Databricks para dejar trazabilidad.

In [0]:
tablas = {
    "silver_peru_compras": "Capa Silver (producción): órdenes de compra reales de PERÚ COMPRAS (ago a oct 2026) que cumplen las reglas de calidad.",
    "quarantine_peru_compras": "Capa Quarantine (producción): órdenes reales observadas, con la lista de reglas incumplidas para auditoría y RCA.",
    "gold_resumen_calidad_peru_compras": "Capa Gold: resumen de la corrida real (registros auditados, aprobados y observados).",
    "gold_scorecard_calidad_peru_compras": "Capa Gold: frecuencia y monto en soles por tipo de anomalía detectada en los datos reales.",
    "silver_peru_compras_test": "PRUEBA NEGATIVA: Silver generado con anomalías inyectadas a propósito. No usar como dato real.",
    "quarantine_peru_compras_test": "PRUEBA NEGATIVA: registros con anomalías inyectadas y aislados por el motor de reglas. No son errores reales.",
    "gold_scorecard_prueba_negativa": "PRUEBA NEGATIVA: scorecard de las anomalías inyectadas y detectadas por tipo de regla."
}
for t, c in tablas.items():
    spark.sql(f"COMMENT ON TABLE workspace.default.{t} IS '{c}'")

for t in ["quarantine_peru_compras", "quarantine_peru_compras_test"]:
    spark.sql(f"ALTER TABLE workspace.default.{t} ALTER COLUMN motivos_rechazo COMMENT 'Lista (array) de reglas de calidad DAMA incumplidas por la orden electrónica.'")

print("Documentación y metadatos registrados correctamente en Unity Catalog.")

Documentación y metadatos registrados correctamente en Unity Catalog.
